In [16]:
# ODIN v11: THE STRATEGIC GROWTH ORACLE (Read-Only CEO Edition)
# Focus: Quality of Earnings, Growth Velocity, and Operational Integrity

import os
import io
import xmlrpc.client
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from dotenv import load_dotenv

# PDF Generation
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image as RLImage
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch

# ================================================================
# 1. READ-ONLY ODOO CONNECTOR
# ================================================================
load_dotenv()
plt.switch_backend('Agg')

class OdooOracle:
    def __init__(self):
        self.url = os.getenv('ODOO_URL', 'https://vendyltd.odoo.com/')
        self.db = os.getenv('ODOO_DB', 'vendyltd')
        self.user = os.getenv('ODOO_USER', 'muktadir@vendy.ltd')
        self.pwd = os.getenv('ODOO_PASSWORD', '205958c9bb841b4bef7e6da4a3afb5b5029cd6ae')
        
        common = xmlrpc.client.ServerProxy(f"{self.url}/xmlrpc/2/common")
        self.uid = common.authenticate(self.db, self.user, self.pwd, {})
        self.models = xmlrpc.client.ServerProxy(f"{self.url}/xmlrpc/2/object")
        print(f"✅ Oracle Authenticated (UID: {self.uid})")

    def read(self, model, domain, fields):
        """Strict search_read only to prevent database modification."""
        return self.models.execute_kw(self.db, self.uid, self.pwd, model, 'search_read', [domain], {'fields': fields})

# ================================================================
# 2. STRATEGIC INTELLIGENCE ENGINE
# ================================================================
class IntelligenceEngine:
    @staticmethod
    def get_survival_metrics(oracle):
        # Analyze Profit vs Cash Gap
        moves = oracle.read('account.move', [('state', '=', 'posted')], ['amount_total', 'move_type', 'payment_state'])
        df = pd.DataFrame(moves)
        
        profit = df[df['move_type'] == 'out_invoice']['amount_total'].sum()
        cash = df[(df['move_type'] == 'out_invoice') & (df['payment_state'] == 'paid')]['amount_total'].sum()
        cash_gap = profit - cash
        
        # Quality of Earnings Ratio (Cash / Profit)
        qoe_ratio = (cash / profit) if profit > 0 else 0
        return {"profit": profit, "cash": cash, "gap": cash_gap, "qoe": qoe_ratio}

    @staticmethod
    def get_operational_risks(oracle):
        # Audit Ghost Assets (Negative Stock)
        ghosts = oracle.read('stock.quant', [('quantity', '<', 0)], ['product_id', 'quantity'])
        
        # Analyze Delivery Bottlenecks
        pickings = oracle.read('stock.picking', [], ['state'])
        df_p = pd.DataFrame(pickings)
        completion_rate = (len(df_p[df_p['state'] == 'done']) / len(df_p)) * 100 if not df_p.empty else 0
        
        return {"ghost_count": len(ghosts), "completion_rate": round(completion_rate, 1)}

# ================================================================
# 3. EXECUTIVE VISUALIZER & PDF ENGINE
# ================================================================
class ExecutiveReporter:
    def __init__(self):
        self.styles = getSampleStyleSheet()
        self.color_navy = colors.HexColor('#1a237e')

    def plot_qoe(self, profit, cash):
        plt.figure(figsize=(5, 3))
        plt.bar(['Paper Profit', 'Real Cash'], [profit, cash], color=['#4caf50', '#f44336'])
        plt.title("QUALITY OF EARNINGS (QoE)", fontweight='bold')
        buf = io.BytesIO()
        plt.savefig(buf, format='png', dpi=150)
        buf.seek(0)
        return buf

    def generate_pdf(self, metrics, risks, filename="CEO_STRATEGIC_ORACLE.pdf"):
        doc = SimpleDocTemplate(filename, pagesize=letter)
        story = []
        
        # Header
        story.append(Paragraph("ODIN STRATEGIC GROWTH ORACLE", self.styles['Title']))
        story.append(Spacer(1, 20))
        
        # Section 1: Financial Health
        story.append(Paragraph("1. FINANCIAL SURVIVAL STATUS", self.styles['Heading2']))
        story.append(RLImage(self.plot_qoe(metrics['profit'], metrics['cash']), width=4*inch, height=2.5*inch))
        
        health_text = f"""<b>Analysis:</b> The business shows a <b>Paper Profit of ${metrics['profit']:,.2f}</b>, 
        but current <b>Real Cash Flow is ${metrics['cash']:,.2f}</b>. <br/><br/>
        <b>QoE Ratio: {metrics['qoe']:.2f}</b> (Critical Threshold: < 0.8). <br/>
        The liquidity gap is currently <b>${metrics['gap']:,.2f}</b>."""
        story.append(Paragraph(health_text, self.styles['Normal']))
        story.append(Spacer(1, 20))
        
        # Section 2: Growth Bottlenecks
        story.append(Paragraph("2. OPERATIONAL GROWTH BOTTLENECKS", self.styles['Heading2']))
        bottleneck_text = f"""• <b>Delivery Completion Rate: {risks['completion_rate']}%</b>. (Growth is blocked by warehouse logistics). <br/>
        • <b>Integrity Warning: {risks['ghost_count']} Ghost Assets</b> detected. (Negative stock levels invalidating company valuation)."""
        story.append(Paragraph(bottleneck_text, self.styles['Normal']))
        
        doc.build(story)
        print(f"✅ Strategic Oracle Report saved as: {filename}")

# ================================================================
# EXECUTION
# ================================================================
if __name__ == "__main__":
    oracle_client = OdooOracle()
    intel = IntelligenceEngine()
    reporter = ExecutiveReporter()
    
    # 1. Gather Insights
    m = intel.get_survival_metrics(oracle_client)
    r = intel.get_operational_risks(oracle_client)
    
    # 2. Output to PDF
    reporter.generate_pdf(m, r)

✅ Oracle Authenticated (UID: 2)
✅ Strategic Oracle Report saved as: CEO_STRATEGIC_ORACLE.pdf
